### TRANSFORMATION WORKFLOW

1. KEEP THE CURRENT VERSION OF EACH PRODUCT
2. NORMALIZE CATEGORY_ID FORMAT
3. JOIN PRODUCTS + CATEGORIES
4. ADD SURROGATE KEY
5. ADD UNKNOWN/DISCONTINUED PLACEHOLDER ROW
6. WRITE INTO GOLD

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType 
from pyspark.sql.window import Window

In [0]:
# 0) LOAD ALL THE PRODUCT RELATED TABLES FROM SILVER LAYER
df_crmp = spark.table("acdproj.silver.crm_products")
df_erpcat = spark.table("acdproj.silver.erp_categories")

In [0]:
# 1) KEEP ONLY THE CURRENT VERSION OF EACH PRODUCT
# crm_products has historical rows per product_key (SCD Type 2 style — price
# changes over time). For this dimension we keep only the current version,
# identified by product_end_date being null.
df_crmp = df_crmp.filter(F.col("product_end_date").isNull())

# 2) NORMALIZE CATEGORY_ID FORMAT (crm uses hyphens, erp uses underscores)
df_crmp = df_crmp.withColumn("category_id", F.regexp_replace(F.col("category_id"), "-", "_"))

In [0]:
# 3) JOIN PRODUCTS + CATEGORIES
dim_products = (
    df_crmp.alias("prod")
    .join(df_erpcat.alias("cat"), F.col("prod.category_id") == F.col("cat.category_id"), "left")
    .select(
        F.col("prod.product_key").alias("product_key"),
        F.col("prod.product_key_short"),
        F.col("prod.product_name"),
        F.col("prod.product_cost"),
        F.col("prod.product_line"),
        F.col("cat.category"),
        F.col("cat.subcategory"),
        F.col("cat.maintenance"),
        F.col("prod.product_start_date"),
        F.col("prod.product_end_date")
    )
)

In [0]:
 # 4) ADD SURROGATE KEY

window_spec = Window.orderBy("product_key")
dim_products = dim_products.withColumn("product_sk", F.row_number().over(window_spec))

# Reorder for readability
dim_products = dim_products.select(
    "product_sk", "product_key", "product_key_short", "product_name", "product_cost",
    "product_line", "category", "subcategory", "maintenance",
    "product_start_date", "product_end_date"
)

dim_products.display()

In [0]:
dim_products.printSchema()

In [0]:
dim_products.display()

In [0]:
# 5) ADD UNKNOWN/DISCONTINUED PLACEHOLDER ROW
# Sales referencing discontinued products (all versions have an end_date) won't
# match any row here. This placeholder gives them a valid foreign key instead
# of a null, so BI tools show an explicit "Unknown" category instead of blanks.
unknown_row = spark.createDataFrame(
    [(-1, "UNKNOWN", "UNKNOWN", "Unknown/Discontinued Product", 0, "n/a",
      "n/a", "n/a", "n/a", None, None)],
    schema=dim_products.schema
)

dim_products = dim_products.unionByName(unknown_row)

dim_products.display()

In [0]:
# 6) WRITE INTO GOLD
dim_products.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("acdproj.gold.dim_products")